# Notebook 01 – Incidenti stradali ISTAT (dataset 41_983)

**1. Download** – Get the data from the ISTAT API (CSV) and save the raw file in `Data/Raw`.

In [1]:
import pandas as pd
import requests
import os
os.makedirs("Data/Raw", exist_ok=True) 

url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}

r = requests.get(url, headers=headers)
print(r.status_code)

if r.status_code == 200:
    with open("Data/Raw/incidenti_istat.csv", "wb") as f:
        f.write(r.content)                                

200


**2. Read the file** – Load and print the CSV.

In [2]:
df = pd.read_csv("Data/Raw/incidenti_istat.csv")
print(df.shape)

(573552, 16)


**3. Columns** – `info()`: only 5 columns matter (`REF_AREA`, `DATA_TYPE`, `RESULT`, `TIME_PERIOD`, `OBS_VALUE`); the others are empty or not needed.


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

**4. Codes** – `value_counts()` counts rows, not accidents: each municipality-year has 3 rows (`ROADACC/9`, `KILLINJ/F`, `KILLINJ/M`).

In [4]:
print(df["DATA_TYPE"].value_counts())
print(df["RESULT"].value_counts())

DATA_TYPE
KILLINJ    382368
ROADACC    191184
Name: count, dtype: int64
RESULT
F    191184
M    191184
9    191184
Name: count, dtype: int64


**5. Meaning of the codes** – Sum of `OBS_VALUE` per code: `9` = accidents, `F` = injured, `M`.

In [5]:
df.groupby(["DATA_TYPE", "RESULT"])["OBS_VALUE"].sum()

DATA_TYPE  RESULT
KILLINJ    F         6709026
           M          101944
ROADACC    9         4779141
Name: OBS_VALUE, dtype: int64

**6. Quality checks** – No duplicate rows and no negative values.

In [7]:
keys = ["REF_AREA", "DATA_TYPE", "RESULT", "TIME_PERIOD"]
print("Duplicate rows:", df.duplicated(subset=keys).sum())
print("Negative values:", (df["OBS_VALUE"] < 0).sum())

Duplicate rows: 0
Negative values: 0


**7. Clean and pivot** – Keep the useful columns and rename the codes.

In [8]:
df_clean = df[["REF_AREA", "RESULT", "TIME_PERIOD", "OBS_VALUE"]].copy()
df_clean["RESULT"] = df_clean["RESULT"].map({"9": "accidents", "F": "injured", "M": "deaths"})
df_wide = df_clean.pivot(
    index=["REF_AREA", "TIME_PERIOD"],
    columns="RESULT",
    values="OBS_VALUE",
).reset_index()
df_wide.head()

RESULT,REF_AREA,TIME_PERIOD,accidents,deaths,injured
0,1001,2001,5,0,10
1,1001,2002,5,0,10
2,1001,2003,4,0,7
3,1001,2004,9,0,13
4,1001,2005,2,0,2


**8. Logic check** – No row has more accidents than injured + deaths.

In [9]:
suspicious = df_wide[df_wide["accidents"] > df_wide["injured"] + df_wide["deaths"]]
print("Suspicious rows:", len(suspicious))

Suspicious rows: 0


**9. Save** – Write the clean table to `Data/Processed/accidents_clean.csv` and read it back to confirm the shape.

In [10]:
os.makedirs("Data/Processed", exist_ok=True)
df_wide.to_csv("Data/Processed/accidents_clean.csv", index=False)
df_check = pd.read_csv("Data/Processed/accidents_clean.csv")
print(df_check.shape)      

(191184, 5)


**10. Municipalities per year** – Municipalities listed, min and max accidents per year.

In [12]:
by_year = df_wide.groupby("TIME_PERIOD").agg(
    num_of_municipalities=("REF_AREA", "count"),    
    min_accidents=("accidents", "min"),
    max_accidents=("accidents", "max"),         
).reset_index()
by_year

,TIME_PERIOD,num_of_municipalities,min_accidents,max_accidents
0,2001,8101,0,22220
1,2002,8102,0,21330
2,2003,8102,0,20426
3,2004,8100,0,23135
4,2005,8101,0,21902
5,2006,8101,0,21452
6,2007,8101,0,19960
7,2008,8101,0,18181
8,2009,8100,0,18561
9,2010,8094,0,18496


**11. Missing municipalities** – 1,562 missing in 2024 vs 1,643 with 0 accidents in 2023

In [13]:
n_2023 = (df_wide["TIME_PERIOD"] == 2023).sum()     
n_2024 = (df_wide["TIME_PERIOD"] == 2024).sum()     
zero_2023 = ((df_wide["TIME_PERIOD"] == 2023) & (df_wide["accidents"] == 0)).sum() 
print("Municipalities listed in 2023:", n_2023)
print("Municipalities listed in 2024:", n_2024)
print("Missing in 2024:", n_2023 - n_2024)
print("Municipalities with 0 accidents in 2023:", zero_2023)

Municipalities listed in 2023: 7901
Municipalities listed in 2024: 6339
Missing in 2024: 1562
Municipalities with 0 accidents in 2023: 1643
